# COMMUN

### Imports et configuration

In [6]:
import os
import yaml
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
from pathlib import Path

### Charger variables d'environnement depuis .env

In [7]:
load_dotenv()

True

In [18]:
try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    ROOT_DIR = Path.cwd().parent

CONFIG_PATH = ROOT_DIR / "config.yml"
print(CONFIG_PATH)

c:\Users\DELL\Documents\vscode_simplon\Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes\config.yml


### Définir le chemin racine du projet (quel que soit le dossier courant)

In [19]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    for section, values in config.items():
        for key, val in values.items():
            if isinstance(val, str) and val.startswith("${"):
                env_var = val.strip("${}")
                config[section][key] = os.getenv(env_var)    
    return config

### Lecture du fichier config.yml

In [20]:
conf = load_config(CONFIG_PATH)


### Connection avec Database

In [ ]:
db_conf = conf["database"]

connection_url = f"postgresql+psycopg2://{db_conf['user']}:{db_conf['password']}@{db_conf['host']}:{db_conf['port']}/{db_conf['db']}"
engine = create_engine(connection_url)

with engine.connect() as conn:

    result = conn.execute(text("SELECT version();"))
    print(result.fetchone())

In [ ]:
api_url = config["api"]["url"]
headers = {"Authorization": f"Bearer {config['api']['key']}"}

response = requests.get(api_url, headers=headers)
response.raise_for_status()
data = response.json()

df = pd.DataFrame(data)
print("✅ Données récupérées :", len(df), "lignes")
df.head()

In [ ]:
api_url = /api/explore/v2.1/catalog/datasets/accidents-corporels-de-la-circulation-millesime/records/?limit=10&offset=0
headers = {"Authorization": f"Bearer {config['api']['key']}"}

response = requests.get(api_url, headers=headers)
response.raise_for_status()
data = response.json()

df = pd.DataFrame(data)
print("✅ Données récupérées :", len(df), "lignes")
df.head()

# AHMED

In [53]:
csv = ROOT_DIR / "data" / "acc_2017.csv"
print(csv)

c:\Users\DELL\Documents\vscode_simplon\Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes\data\acc_2017.csv


# ROMAIN

In [23]:
API_CONFIG = {
    'base_url' : 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/accidents-corporels-de-la-circulation-millesime/records',
    'limit_per_request' : 100,
    'max_records' : 1000,
    'timeout' : 30
}

print(f"API: {API_CONFIG['base_url'][:70]}...")

API: https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/acci...


In [25]:
def extraire_accidents_api(max_records=None):
    """
    Fonction pour extraire les accidents depuis l'API
    
    Paramètres:
        max_records (int): Le nombre maximum d'accidents à extraire. Si None, tous les accidents seront extraits.
    
    Return:
        list: Une liste de dictionnaires représentant les accidents extraîts.
    """

print("=" * 80)
print("EXTRACTION DES DONNÉES")
print("=" * 80)

all_records = []
offset = 0
limit = API_CONFIG['limit_per_request']

try:
    #première requête pour connaitre le total
    print("Récupération du nombre total...")
    response = requests.get(
        API_CONFIG['base_url'],
        params={'limit': 1},
        timeout=API_CONFIG['timeout']
    )
    response.raise_for_status()
    data = response.json()
    total_count = data.get('total_count', 0)
    
    print(f"Total disponible: {total_count:,} enregistrements")
    print(data)       
except requests.RequestException as e:
    print(f"\n✗ Erreur lors de l'extraction: {e}")
    raise

EXTRACTION DES DONNÉES
Récupération du nombre total...
Total disponible: 475,911 enregistrements
{'total_count': 475911, 'results': [{'num_acc': '201700009715', 'datetime': '2017-05-28T16:50:00+00:00', 'nom_com': None, 'an': '2017', 'mois': '05', 'jour': '28', 'hrmn': '18:50', 'lum': 'Plein jour', 'agg': 'En agglomération', 'int': '3', 'atm': 'Normale', 'col': 'Deux véhicules – par le coté', 'dep': '13', 'com': '055', 'insee': '13055', 'adr': '6 Av Alexandre  Ansaldi', 'lat': '4333582', 'long': '0539866', 'code_postal': None, 'num': '6', 'coordonnees': {'lon': 2.911777, 'lat': 42.686216}, 'pr': None, 'surf': 'normale', 'v1': None, 'circ': 'Bidirectionnelle', 'vosp': None, 'env1': '00', 'voie': '4', 'larrout': 120, 'v2': None, 'lartpc': 25, 'nbv': 4, 'catr': 'Route Départementale', 'pr1': None, 'plan': 'Partie rectiligne', 'prof': 'Plat', 'infra': None, 'situ': 'Sur chaussée', 'an_nais': ['1998', '1966'], 'sexe': ['Masculin', 'Masculin'], 'actp': ['Se déplaçant', 'Se déplaçant'], 'grav'

# LOUNES

# ZOUBIR